In [ ]:
import pandas as pd
import numpy as np
import shap
import xgboost as xgb
import matplotlib.pyplot as plt

# Step 1: Load the Excel dataset
file_path = "/content/dataset.xlsx"  # Replace with your actual file path
sheet_name = "Sheet1"         # Replace with your actual sheet name


df = pd.read_excel(file_path, sheet_name=sheet_name)

# Step 2: Clean column names to remove XGBoost-incompatible characters
df.columns = df.columns.str.replace(r"[\[\]<>]", "", regex=True)

# Step 3: Fill missing values with column means
df = df.fillna(df.mean(numeric_only=True))

# Step 4: Separate features and target
target_column = "IVIGRKD"  # Replace with your actual target column
X_full = df.drop(columns=[target_column])
y = df[target_column]

# Step 5: Train an XGBoost regression model
model = xgb.XGBRegressor()
model.fit(X_full, y)

# Step 6: Compute SHAP values
explainer = shap.Explainer(model, X_full)
shap_values = explainer(X_full)

# Step 7: Calculate mean absolute SHAP values for feature importance
shap_importance = pd.DataFrame({
    'Feature': X_full.columns,
    'SHAP Importance': np.abs(shap_values.values).mean(axis=0)
}).sort_values(by="SHAP Importance", ascending=False)

# Step 8: Extract the top 15 features
top_14_features = shap_importance['Feature'].head(14).tolist()
X= X_full[top_14_features]

# Step 9: Save to Excel
X.to_excel("X_top14_SHAP.xlsx", index=False)

# Step 10: Optional - Display and plot
print("Top 14 features based on SHAP values:")
print(shap_importance.head(14))

shap.summary_plot(shap_values[:, top_14_features], X_full[top_14_features])



In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers timm
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2
!pip install transformers==4.38.2
!pip install autogluon.multimodal

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)  # Sensitivity = Recall (Positive class)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)

    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return [acc, sen, spec, prec, f1, auc]

In [ ]:
models = {
    "LR": LogisticRegression(max_iter=1000),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "GB": GradientBoostingClassifier(),
    "LGBM": LGBMClassifier()
}

In [ ]:
results = {}

for name, model in models.items():
    results[name] = evaluate_model(model, X_train, X_test, y_train, y_test)

results_df = pd.DataFrame(results,
                          index=["Accuracy", "Sensitivity", "Specificity",
                                 "Precision", "F1-score", "AUC"])

print(results_df.T)

In [ ]:
X_train_cnn = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn  = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(82, 3, activation='relu', input_shape=(X.shape[1],1)),
    tf.keras.layers.MaxPooling1D(1),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Conv1D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling1D(1),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Conv1D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling1D(1),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

history = model.fit(X_train_cnn, y_train,
                    epochs=10,
                    validation_split=0.1,
                    verbose=1)

In [ ]:
y_prob_cnn = model.predict(X_test_cnn).ravel()
y_pred_cnn = (y_prob_cnn > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred_cnn)
sen = recall_score(y_test, y_pred_cnn)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_cnn).ravel()
spec = tn / (tn + fp)

prec = precision_score(y_test, y_pred_cnn)
f1 = f1_score(y_test, y_pred_cnn)
auc = roc_auc_score(y_test, y_prob_cnn)

results["CNN"] = [acc, sen, spec, prec, f1, auc]

results_df = pd.DataFrame(results,
                          index=["Accuracy", "Sensitivity", "Specificity",
                                 "Precision", "F1-score", "AUC"])

print(results_df.T)

In [ ]:
!pip install imbalanced-learn lightgbm

In [ ]:
# ======================================================
# Multiple Sampling Methods + All Models + Save Results
# ======================================================

import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

# -----------------------------
# Define Sampling Methods
# -----------------------------
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RUS": RandomUnderSampler(random_state=42)
}

# -----------------------------
# Define Models
# -----------------------------
models = {
    "LR": LogisticRegression(max_iter=1000),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "GB": GradientBoostingClassifier(),
    "LGBM": LGBMClassifier(),
    "NB": GaussianNB()
}

# -----------------------------
# Evaluation Function
# -----------------------------
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)

    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return acc, sen, spec, prec, f1, auc


# -----------------------------
# Run Experiments
# -----------------------------
all_results = []

for sampler_name, sampler in samplers.items():

    print(f"\nRunning Sampling Method: {sampler_name}")

    # Apply sampling only on training data
    X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)

    for model_name, model in models.items():

        acc, sen, spec, prec, f1, auc = evaluate_model(
            model,
            X_train_res,
            X_test,
            y_train_res,
            y_test
        )

        all_results.append([
            sampler_name,
            model_name,
            acc,
            sen,
            spec,
            prec,
            f1,
            auc
        ])

# -----------------------------
# Save Results
# -----------------------------
results_df = pd.DataFrame(
    all_results,
    columns=[
        "Sampling Method",
        "Model",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-score",
        "AUC"
    ]
)

# Display Results
print("\nFinal Results:")
print(results_df)

# Save to CSV
results_df.to_csv("sampling_model_results.csv", index=False)

In [ ]:
# ==========================================================
# CNN + SMOTE / ADASYN / RUS + Save Results
# ==========================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

# -----------------------------
# Define Sampling Methods
# -----------------------------
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RUS": RandomUnderSampler(random_state=42)
}

# -----------------------------
# CNN Model Function
# -----------------------------
def create_cnn_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(82, 3, activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling1D(1),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(1),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(1),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

    return model


# -----------------------------
# Run Experiments
# -----------------------------
cnn_results = []

for sampler_name, sampler in samplers.items():

    print(f"\nRunning CNN with {sampler_name}")

    # Apply sampling on training set only
    X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)

    # Convert to numpy
    X_train_res = np.array(X_train_res)
    X_test_np = np.array(X_test)

    # Reshape for CNN
    X_train_res = X_train_res.reshape(X_train_res.shape[0], X_train_res.shape[1], 1)
    X_test_cnn = X_test_np.reshape(X_test_np.shape[0], X_test_np.shape[1], 1)

    # Create fresh model each time
    model = create_cnn_model((X_train_res.shape[1], 1))

    # Train
    model.fit(X_train_res, y_train_res,
              epochs=10,
              batch_size=32,
              verbose=0)

    # Predict
    y_prob = model.predict(X_test_cnn).ravel()
    y_pred = (y_prob > 0.5).astype(int)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)

    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    cnn_results.append([
        sampler_name,
        acc,
        sen,
        spec,
        prec,
        f1,
        auc
    ])

# -----------------------------
# Save Results
# -----------------------------
cnn_results_df = pd.DataFrame(
    cnn_results,
    columns=[
        "Sampling Method",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-score",
        "AUC"
    ]
)

print("\nCNN Results:")
print(cnn_results_df)

cnn_results_df.to_csv("cnn_sampling_results.csv", index=False)

In [ ]:
print(X.shape)
print(X_train.shape)

In [ ]:
print(y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
imp = SimpleImputer(missing_values=np.nan, strategy='mean')
imp.fit(X_train)
X_train=imp.transform(X_train)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
imp.fit(X_test)
X_test=imp.transform(X_test)
X_test = scaler.transform(X_test)

In [ ]:
newdataset=X

In [ ]:
NLPdataset=pd.DataFrame()

In [ ]:
# List of your 15 selected features
features = [
    "days of fever", "PLT(10^9/L)", "Alb/Glb", "LYM(10^9/L)", "weight",
    "ALT/AST", "Mg2+", "RBC(10^12/L)", "CK", "CO2-CP",
    "Proximal RCA(mm)", "Hct%", "P-LCR%", "Eos(10^9/L)"
]

# Loop through and print describe for each
for feature in features:
    print(f"\nFeature: {feature}")
    print(newdataset[feature].describe())


In [ ]:
def condition_days_of_fever(x):
    if x < 3:
        return "very short fever"
    elif 3 <= x < 5:
        return "short fever"
    elif 5 <= x < 7:
        return "moderate fever"
    elif 7 <= x < 10:
        return "long fever"
    else:
        return "very long fever"


In [ ]:
NLPdataset['days of fever'] = newdataset['days of fever'].apply(condition_days_of_fever)


In [ ]:
def condition_PLT(x):
    if x < 150:
        return "very low platelet count"
    elif 150 <= x < 300:
        return "low platelet count"
    elif 300 <= x < 450:
        return "normal platelet count"
    elif 450 <= x < 600:
        return "elevated platelet count"
    else:
        return "very high platelet count"


In [ ]:
NLPdataset['PLT(10^9/L)'] = newdataset['PLT(10^9/L)'].apply(condition_PLT)


In [ ]:
def condition_Alb_Glb(x):
    if x < 1.0:
        return "very low Alb/Glb ratio"
    elif 1.0 <= x < 1.3:
        return "low Alb/Glb ratio"
    elif 1.3 <= x < 1.7:
        return "normal Alb/Glb ratio"
    elif 1.7 <= x < 2.0:
        return "elevated Alb/Glb ratio"
    else:
        return "high Alb/Glb ratio"


In [ ]:
NLPdataset['Alb/Glb'] = newdataset['Alb/Glb'].apply(condition_Alb_Glb)


In [ ]:
def condition_LYM(x):
    if x < 1.5:
        return "very low lymphocyte count"
    elif 1.5 <= x < 3.0:
        return "low lymphocyte count"
    elif 3.0 <= x < 5.5:
        return "normal lymphocyte count"
    elif 5.5 <= x < 8.0:
        return "elevated lymphocyte count"
    else:
        return "very high lymphocyte count"


In [ ]:
NLPdataset['LYM(10^9/L)'] = newdataset['LYM(10^9/L)'].apply(condition_LYM)


In [ ]:
def condition_weight(x):
    if x < 6:
        return "very low weight"
    elif 6 <= x < 9:
        return "low weight"
    elif 9 <= x < 12:
        return "normal weight"
    elif 12 <= x < 15:
        return "slightly high weight"
    else:
        return "high weight"


In [ ]:
def condition_ALT_AST(x):
    if x < 0.5:
        return "very low ALT/AST ratio"
    elif 0.5 <= x < 1.0:
        return "low ALT/AST ratio"
    elif 1.0 <= x < 2.0:
        return "normal ALT/AST ratio"
    elif 2.0 <= x < 4.0:
        return "elevated ALT/AST ratio"
    else:
        return "high ALT/AST ratio"


In [ ]:
def condition_Mg2(x):
    if x < 0.9:
        return "low magnesium level"
    elif 0.9 <= x <= 1.2:
        return "normal magnesium level"
    elif 1.2 < x <= 2.5:
        return "slightly elevated magnesium level"
    else:
        return "abnormally high magnesium level"


In [ ]:
NLPdataset['weight'] = newdataset['weight'].apply(condition_weight)
NLPdataset['ALT/AST'] = newdataset['ALT/AST'].apply(condition_ALT_AST)
NLPdataset['Mg2+'] = newdataset['Mg2+'].apply(condition_Mg2)


In [ ]:
def condition_RBC(x):
    if x < 3.5:
        return "low red blood cell count"
    elif 3.5 <= x < 4.2:
        return "normal red blood cell count"
    elif 4.2 <= x < 5.0:
        return "elevated red blood cell count"
    else:
        return "high red blood cell count"


In [ ]:
def condition_CK(x):
    if x == 0:
        return "no CK detected"
    elif 0 < x < 30:
        return "low CK level"
    elif 30 <= x < 80:
        return "normal CK level"
    elif 80 <= x < 200:
        return "elevated CK level"
    else:
        return "very high CK level"


In [ ]:
def condition_CO2_CP(x):
    if x < 10:
        return "very low CO2 combining power"
    elif 10 <= x < 18:
        return "low CO2 combining power"
    elif 18 <= x < 25:
        return "normal CO2 combining power"
    else:
        return "elevated CO2 combining power"


In [ ]:
NLPdataset['RBC(10^12/L)'] = newdataset['RBC(10^12/L)'].apply(condition_RBC)
NLPdataset['CK'] = newdataset['CK'].apply(condition_CK)
NLPdataset['CO2-CP'] = newdataset['CO2-CP'].apply(condition_CO2_CP)


In [ ]:
def condition_Proximal_RCA(x):
    if x < 2.0:
        return "small RCA diameter"
    elif 2.0 <= x < 2.5:
        return "normal RCA diameter"
    elif 2.5 <= x < 3.5:
        return "mildly dilated RCA"
    else:
        return "severely dilated RCA"


In [ ]:
def condition_Hct(x):
    if x < 28:
        return "very low hematocrit"
    elif 28 <= x < 32:
        return "low hematocrit"
    elif 32 <= x < 36:
        return "normal hematocrit"
    else:
        return "high hematocrit"


In [ ]:
def condition_PLCR(x):
    if x < 15:
        return "very low P-LCR"
    elif 15 <= x < 22:
        return "low P-LCR"
    elif 22 <= x < 28:
        return "normal P-LCR"
    else:
        return "high P-LCR"


In [ ]:
def condition_Eos(x):
    if x == 0:
        return "no eosinophils"
    elif 0 < x < 0.15:
        return "low eosinophil count"
    elif 0.15 <= x < 0.6:
        return "normal eosinophil count"
    else:
        return "elevated eosinophil count"


In [ ]:
NLPdataset['Proximal RCA(mm)'] = newdataset['Proximal RCA(mm)'].apply(condition_Proximal_RCA)
NLPdataset['Hct%'] = newdataset['Hct%'].apply(condition_Hct)
NLPdataset['P-LCR%'] = newdataset['P-LCR%'].apply(condition_PLCR)
NLPdataset['Eos(10^9/L)'] = newdataset['Eos(10^9/L)'].apply(condition_Eos)


In [ ]:
y_labeled = y.replace({0: "IVIG responsive", 1: "IVIG resistant"})


In [ ]:
def create_sentence(row):
    return (
        f"The patient had {row['PLT(10^9/L)']}, "
        f"{row['ALT/AST']} ratio, "
        f"{row['LYM(10^9/L)']} lymphocyte count, "
        f"{row['RBC(10^12/L)']} red blood cell count, "
        f"{row['CK']} CK level, "
        f"{row['CO2-CP']} CO2 combining power, "
        f"{row['Proximal RCA(mm)']}, "
        f"{row['Hct%']} hematocrit, "
        f"{row['P-LCR%']} P-LCR, "
        f"{row['Eos(10^9/L)']} eosinophil count, "

        f"{row['Mg2+']} magnesium level, "
        f"{row['weight']} weight, "
        f"{row['days of fever']} duration of fever."
    )

# Apply to entire DataFrame
sentds = pd.DataFrame()
sentds['sentence'] = NLPdataset.apply(create_sentence, axis=1)
X = sentds


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,y_labeled,test_size=0.20,random_state=5)

In [ ]:
docs=x_train['sentence'].tolist()

rdocs=x_test['sentence'].tolist()
labels = y_train.tolist()
rlabel = y_test.tolist()


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": docs, "label": labels})
test_dataset = Dataset.from_dict({"text": rdocs, "label": rlabel})


In [ ]:
# ==========================================================
# CELL 1: Extract Sentence Transformer Embeddings
#         + Save in Colab Local (/content/)
# ==========================================================

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load MiniLM-L3-v2
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/paraphrase-MiniLM-L3-v2"
)

model = AutoModel.from_pretrained(
    "sentence-transformers/paraphrase-MiniLM-L3-v2"
)

model.to(device)
model.eval()


# -------- Mean Pooling --------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )


# -------- Embedding Function --------
def get_embeddings(texts, batch_size=16):

    all_embeddings = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        encoded_input = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=256
        ).to(device)

        with torch.no_grad():
            model_output = model(**encoded_input)

        sentence_embeddings = mean_pooling(
            model_output,
            encoded_input['attention_mask']
        )

        all_embeddings.append(sentence_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)


# -------- Generate Embeddings --------
train_embeddings_np = get_embeddings(docs)
test_embeddings_np  = get_embeddings(rdocs)

print("Train Shape:", train_embeddings_np.shape)
print("Test Shape :", test_embeddings_np.shape)


# ==========================================================
# FIX LABELS PROPERLY
# ==========================================================

y_train = np.array(y_train)
y_test  = np.array(y_test)

# If labels are strings → map them
if y_train.dtype == object:
    print("String labels detected. Encoding them...")

    label_mapping = {
        "IVIG responsive": 0,
        "IVIG resistant": 1
    }

    y_train = np.array([label_mapping[label] for label in y_train])
    y_test  = np.array([label_mapping[label] for label in y_test])

# Ensure integer type
y_train = y_train.astype(int)
y_test  = y_test.astype(int)

print("Unique Train Labels:", np.unique(y_train))
print("Unique Test Labels :", np.unique(y_test))


# -------- Save in Colab Local (/content/) --------
np.save("train_sentence_embeddings.npy", train_embeddings_np)
np.save("test_sentence_embeddings.npy", test_embeddings_np)
np.save("train_labels.npy", y_train)
np.save("test_labels.npy", y_test)

print("✅ Files saved in Colab local storage.")

In [ ]:
# ==========================================================
# CELL 2: Oversampling + ML Models + Evaluation
#         (Uses Local Colab Files)
# ==========================================================

import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix


# ----------------------------------------------------------
# 1️⃣ Load Saved Embeddings & Labels
# ----------------------------------------------------------
train_embeddings_np = np.load("train_sentence_embeddings.npy")
test_embeddings_np  = np.load("test_sentence_embeddings.npy")
y_train = np.load("train_labels.npy")
y_test  = np.load("test_labels.npy")

print("Loaded Shapes:")
print("Train:", train_embeddings_np.shape)
print("Test :", test_embeddings_np.shape)


# ----------------------------------------------------------
# 2️⃣ Define Sampling Methods
# ----------------------------------------------------------
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RUS": RandomUnderSampler(random_state=42)
}


# ----------------------------------------------------------
# 3️⃣ Define ML Models
# ----------------------------------------------------------
models = {
    "LR": LogisticRegression(max_iter=1000),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "GB": GradientBoostingClassifier(),
    "LGBM": LGBMClassifier(),
    "NB": GaussianNB()
}


# ----------------------------------------------------------
# 4️⃣ Evaluation Function
# ----------------------------------------------------------
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)  # Sensitivity

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)

    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return acc, sen, spec, prec, f1, auc


# ----------------------------------------------------------
# 5️⃣ Run Experiments
# ----------------------------------------------------------
all_results = []

for sampler_name, sampler in samplers.items():

    print(f"\nApplying {sampler_name}...")

    X_resampled, y_resampled = sampler.fit_resample(
        train_embeddings_np,
        y_train
    )

    for model_name, model in models.items():

        acc, sen, spec, prec, f1, auc = evaluate_model(
            model,
            X_resampled,
            test_embeddings_np,
            y_resampled,
            y_test
        )

        all_results.append([
            sampler_name,
            model_name,
            acc,
            sen,
            spec,
            prec,
            f1,
            auc
        ])


# ----------------------------------------------------------
# 6️⃣ Create Results Table
# ----------------------------------------------------------
results_df = pd.DataFrame(
    all_results,
    columns=[
        "Sampling Method",
        "Model",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-score",
        "AUC"
    ]
)

print("\nFinal Results:")
print(results_df)


# ----------------------------------------------------------
# 7️⃣ Save Results (Visible in Left Panel)
# ----------------------------------------------------------
results_df.to_csv("embedding_sampling_model_results.csv", index=False)

print("✅ Results saved as embedding_sampling_model_results.csv")

In [ ]:
# ==========================================================
# FT-TRANSFORMER EMBEDDING EXTRACTION (AutoGluon)
# ==========================================================

!pip install autogluon -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from autogluon.multimodal import MultiModalPredictor

# ----------------------------------------------------------
# 1️⃣ Load Dataset
# ----------------------------------------------------------
file_path = "/content/dataset.xlsx"
sheet_name = "Sheet1"
df = pd.read_excel(file_path, sheet_name=sheet_name)

df.columns = df.columns.str.replace(r"[\[\]<>]", "", regex=True)
df = df.fillna(df.mean(numeric_only=True))

target_column = "IVIGRKD"

X_full = df.drop(columns=[target_column])
y = df[target_column].astype(int)  # ensure numeric

# ----------------------------------------------------------
# 2️⃣ Train-Test Split
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=0
)

train_df = X_train.copy()
train_df["target"] = y_train.values

test_df = X_test.copy()
test_df["target"] = y_test.values

# ----------------------------------------------------------
# 3️⃣ Train FT-Transformer via AutoGluon
# ----------------------------------------------------------
predictor = MultiModalPredictor(
    label="target",
    problem_type="binary",
    eval_metric="f1"
)

predictor.fit(train_df, time_limit=300)

# ----------------------------------------------------------
# 4️⃣ Extract Embeddings
# ----------------------------------------------------------
ft_train_embeddings = predictor.extract_embedding(
    train_df.drop(columns=["target"]),
    as_pandas=False
)

ft_test_embeddings = predictor.extract_embedding(
    test_df.drop(columns=["target"]),
    as_pandas=False
)

print("FT Train Embedding Shape:", ft_train_embeddings.shape)
print("FT Test Embedding Shape :", ft_test_embeddings.shape)

# ----------------------------------------------------------
# 5️⃣ Save in Colab Local
# ----------------------------------------------------------
np.save("train_ft_embeddings.npy", ft_train_embeddings)
np.save("test_ft_embeddings.npy", ft_test_embeddings)
np.save("train_labels_ft.npy", y_train.values)
np.save("test_labels_ft.npy", y_test.values)

print("✅ FT-Transformer embeddings saved in Colab local storage.")

In [ ]:
# ==========================================================
# FUSION: Sentence + FT Embeddings
# + Oversampling + ML Models + Evaluation
# ==========================================================

import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix


# ----------------------------------------------------------
# 1️⃣ Load All Embeddings
# ----------------------------------------------------------
sent_train = np.load("train_sentence_embeddings.npy")
sent_test  = np.load("test_sentence_embeddings.npy")

ft_train = np.load("train_ft_embeddings.npy")
ft_test  = np.load("test_ft_embeddings.npy")

y_train = np.load("train_labels.npy")
y_test  = np.load("test_labels.npy")

print("Sentence Train Shape:", sent_train.shape)
print("FT Train Shape      :", ft_train.shape)


# ----------------------------------------------------------
# 2️⃣ Concatenate (Fusion)
# ----------------------------------------------------------
fusion_train = np.concatenate([sent_train, ft_train], axis=1)
fusion_test  = np.concatenate([sent_test, ft_test], axis=1)

print("Fusion Train Shape:", fusion_train.shape)
print("Fusion Test Shape :", fusion_test.shape)


# ----------------------------------------------------------
# 3️⃣ Sampling Methods
# ----------------------------------------------------------
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RUS": RandomUnderSampler(random_state=42)
}


# ----------------------------------------------------------
# 4️⃣ ML Models
# ----------------------------------------------------------
models = {
    "LR": LogisticRegression(max_iter=1000),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "GB": GradientBoostingClassifier(),
    "LGBM": LGBMClassifier(),
    "NB": GaussianNB()
}


# ----------------------------------------------------------
# 5️⃣ Evaluation Function
# ----------------------------------------------------------
def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)

    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return acc, sen, spec, prec, f1, auc


# ----------------------------------------------------------
# 6️⃣ Run Experiments
# ----------------------------------------------------------
all_results = []

for sampler_name, sampler in samplers.items():

    print(f"\nApplying {sampler_name} on Fusion Embeddings")

    X_resampled, y_resampled = sampler.fit_resample(
        fusion_train,
        y_train
    )

    for model_name, model in models.items():

        acc, sen, spec, prec, f1, auc = evaluate_model(
            model,
            X_resampled,
            fusion_test,
            y_resampled,
            y_test
        )

        all_results.append([
            sampler_name,
            model_name,
            acc,
            sen,
            spec,
            prec,
            f1,
            auc
        ])


# ----------------------------------------------------------
# 7️⃣ Save Results
# ----------------------------------------------------------
results_df = pd.DataFrame(
    all_results,
    columns=[
        "Sampling Method",
        "Model",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-score",
        "AUC"
    ]
)

print("\nFinal Fusion Results:")
print(results_df)

results_df.to_csv("fusion_embedding_results.csv", index=False)

print("✅ Fusion results saved.")# =========================
# Bootstrapping for CI
# =========================
def bootstrap_ci(y_true, y_pred, metric_func, n_bootstrap=1000):

    scores = []

    for i in range(n_bootstrap):
        idx = resample(range(len(y_true)))
        score = metric_func(y_true[idx], y_pred[idx])
        scores.append(score)

    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)

    return lower, upper

# =========================
# Decision Curve Function
# =========================
def decision_curve(y_true, probs, thresholds):

    net_benefit = []

    N = len(y_true)

    for pt in thresholds:

        preds = (probs >= pt).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

        nb = (tp/N) - (fp/N)*(pt/(1-pt))

        net_benefit.append(nb)

    return net_benefit


# =========================
# Store results
# =========================
results = []

plt.figure(figsize=(6,5))

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    sens = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn/(tn+fp)

    brier = brier_score_loss(y_test, y_prob)

    # CI
    acc_ci = bootstrap_ci(y_test, y_pred, accuracy_score)
    sens_ci = bootstrap_ci(y_test, y_pred, recall_score)
    f1_ci = bootstrap_ci(y_test, y_pred, f1_score)

    results.append({
        "Model":name,
        "Accuracy":acc,
        "Accuracy CI":acc_ci,
        "Sensitivity":sens,
        "Sensitivity CI":sens_ci,
        "Specificity":spec,
        "F1":f1,
        "F1 CI":f1_ci,
        "Brier Score":brier
    })


    # ======================
    # Calibration Curve
    # ======================

    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)

    plt.plot(prob_pred, prob_true, marker='o', label=name)


# Perfect calibration
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel("Predicted Probability")
plt.ylabel("Observed Probability")
plt.title("Calibration Curve")
plt.legend()
plt.show()


# =========================
# Decision Curve Analysis
# =========================

thresholds = np.linspace(0.01,0.99,100)

plt.figure(figsize=(6,5))

for name, model in models.items():

    y_prob = model.predict_proba(X_test)[:,1]

    nb = decision_curve(y_test, y_prob, thresholds)

    plt.plot(thresholds, nb, label=name)

plt.xlabel("Threshold Probability")
plt.ylabel("Net Benefit")
plt.title("Decision Curve Analysis")
plt.legend()
plt.show()


# =========================
# Print results
# =========================

results_df = pd.DataFrame(results)

print(results_df)

results_df.to_excel("ADASYN_Model_Results_with_CI_Brier.xlsx", index=False)

In [ ]:
# ==========================================================
# FUSION + SAMPLING + CNN MODEL
# ==========================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix


# ----------------------------------------------------------
# 1️⃣ Load Embeddings
# ----------------------------------------------------------
sent_train = np.load("train_sentence_embeddings.npy")
sent_test  = np.load("test_sentence_embeddings.npy")

ft_train = np.load("train_ft_embeddings.npy")
ft_test  = np.load("test_ft_embeddings.npy")

y_train = np.load("train_labels.npy")
y_test  = np.load("test_labels.npy")


# ----------------------------------------------------------
# 2️⃣ Concatenate (Fusion)
# ----------------------------------------------------------
fusion_train = np.concatenate([sent_train, ft_train], axis=1)
fusion_test  = np.concatenate([sent_test, ft_test], axis=1)

print("Fusion Shape:", fusion_train.shape)


# ----------------------------------------------------------
# 3️⃣ Sampling Methods
# ----------------------------------------------------------
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RUS": RandomUnderSampler(random_state=42)
}


# ----------------------------------------------------------
# 4️⃣ CNN Model Definition
# ----------------------------------------------------------
def create_cnn_model(input_shape):

    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(128, 3, activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Conv1D(64, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model


# ----------------------------------------------------------
# 5️⃣ Run Experiments
# ----------------------------------------------------------
cnn_results = []

for sampler_name, sampler in samplers.items():

    print(f"\nApplying {sampler_name} + CNN")

    # Apply sampling
    X_resampled, y_resampled = sampler.fit_resample(
        fusion_train,
        y_train
    )

    # Reshape for CNN
    X_resampled = X_resampled.reshape(X_resampled.shape[0], X_resampled.shape[1], 1)
    fusion_test_cnn = fusion_test.reshape(fusion_test.shape[0], fusion_test.shape[1], 1)

    # Create fresh model
    model = create_cnn_model((X_resampled.shape[1], 1))

    # Train
    model.fit(
        X_resampled,
        y_resampled,
        epochs=15,
        batch_size=32,
        verbose=0
    )

    # Predict
    y_prob = model.predict(fusion_test_cnn).ravel()
    y_pred = (y_prob > 0.5).astype(int)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)

    prec = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    cnn_results.append([
        sampler_name,
        acc,
        sen,
        spec,
        prec,
        f1,
        auc
    ])


# ----------------------------------------------------------
# 6️⃣ Save Results
# ----------------------------------------------------------
cnn_results_df = pd.DataFrame(
    cnn_results,
    columns=[
        "Sampling Method",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-score",
        "AUC"
    ]
)

print("\nFusion + CNN Results:")
print(cnn_results_df)

cnn_results_df.to_csv("fusion_cnn_results.csv", index=False)

print("✅ Fusion CNN results saved.")
# =========================
# Models
# =========================
models = {
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "LightGBM": LGBMClassifier()
}

